In [ ]:
!pip install torchmetrics

In [ ]:
!pip install torchmetrics[image] torch-fidelity

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchmetrics.image.fid import FrechetInceptionDistance
import gc

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Used device: {DEVICE}")

os.makedirs("vae_results", exist_ok=True)

In [ ]:
class BetaVAE(nn.Module):
    def __init__(self, latent_dim=64):
        super(BetaVAE, self).__init__()
        self.latent_dim = latent_dim
        
        # Enkoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1), # 32x32
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), # 16x16
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), # 8x8
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1), # 4x4
            nn.ReLU(),
            nn.Flatten()
        )
        
        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(256 * 4 * 4, latent_dim)
        
        # Dekoder
        self.decoder_input = nn.Linear(latent_dim, 256 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), # 8x8
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), # 16x16
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), # 32x32
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1), # 64x64
            nn.Sigmoid() # Zakres [0, 1]
        )
        
    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
        
    def decode(self, z):
        h = self.decoder_input(z)
        h = h.view(-1, 256, 4, 4)
        return self.decoder(h)
        
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z)
        return recon_x, mu, logvar

def beta_vae_loss(recon_x, x, mu, logvar, beta):
    recon_loss = F.mse_loss(recon_x, x, reduction='sum')
    
    kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    total_loss = recon_loss + beta * kld_loss
    
    batch_size = x.size(0)
    return total_loss / batch_size, recon_loss / batch_size, kld_loss / batch_size

In [ ]:
def train_and_evaluate(config, seed, train_loader, val_loader, epochs=20, phase_name="phase"):
    set_seed(seed)
    
    model = BetaVAE(latent_dim=config['d']).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    
    fid_metric = FrechetInceptionDistance(feature=64).to(DEVICE)
    
    history = []
    best_fid = float('inf')
    best_weights = None
    
    csv_filename = f"vae_results/{phase_name}_history_beta{config['beta']}_d{config['d']}_lr{config['lr']}_seed{seed}.csv"
    
    print(f"\n--- Training start: {phase_name.upper()} | Beta={config['beta']}, d={config['d']}, LR={config['lr']}, Seed={seed} ---")
    
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_recon, train_kld = 0.0, 0.0, 0.0
        
        for batch_idx, (data, _) in enumerate(train_loader):
            data = data.to(DEVICE)
            optimizer.zero_grad()
            
            recon_batch, mu, logvar = model(data)
            loss, recon, kld = beta_vae_loss(recon_batch, data, mu, logvar, config['beta'])
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_recon += recon.item()
            train_kld += kld.item()
            
        avg_loss = train_loss / len(train_loader)
        avg_recon = train_recon / len(train_loader)
        avg_kld = train_kld / len(train_loader)
        
        model.eval()
        fid_metric.reset()
        
        with torch.no_grad():
            for data, _ in val_loader:
                real_imgs = data.to(DEVICE)
                
                real_imgs_uint8 = (real_imgs * 255).byte()
                fid_metric.update(real_imgs_uint8, real=True)
                
                z = torch.randn(real_imgs.size(0), config['d']).to(DEVICE)
                fake_imgs = model.decode(z)
                fake_imgs_uint8 = (fake_imgs * 255).byte()
                
                fid_metric.update(fake_imgs_uint8, real=False)
                
        val_fid = fid_metric.compute().item()
        
        if val_fid < best_fid:
            best_fid = val_fid
            best_weights = {k: v.cpu() for k, v in model.state_dict().items()}
            
        print(f"Epoch {epoch}/{epochs} | Loss: {avg_loss:.4f} (Recon: {avg_recon:.4f}, KLD: {avg_kld:.4f}) | Val FID: {val_fid:.2f}")
        
        history.append({
            'epoch': epoch,
            'train_loss': avg_loss,
            'train_recon': avg_recon,
            'train_kld': avg_kld,
            'val_fid': val_fid
        })
        
        pd.DataFrame(history).to_csv(csv_filename, index=False)
        
    del model, optimizer, fid_metric
    torch.cuda.empty_cache()
    gc.collect()
    
    return best_fid, best_weights

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/crawford/cat-dataset/cats"

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(), # Automatycznie konwertuje do [0, 1]
])

full_dataset = ImageFolder(root=DATASET_PATH, transform=transform)

val_size = int(0.1 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
SEEDS = [42, 142, 242]
EPOCHS = 30

In [ ]:
betas_to_test = [1.0, 4.0, 8.0]
baseline_d = 64
baseline_lr = 1e-3

print("\n" + "="*50)
print(" Phase 1: Optimisation Beta (KLD)")
print("="*50)

best_beta = None
best_beta_fid = float('inf')
best_beta_weights = None
phase1_results = []

for beta in betas_to_test:
    config = {'beta': beta, 'd': baseline_d, 'lr': baseline_lr}
    fid_scores = []
    
    best_seed_fid = float('inf')
    best_seed_weights = None
    
    for seed in SEEDS:
        # Dodano argument phase_name="phase1"
        fid, weights = train_and_evaluate(config, seed, train_loader, val_loader, epochs=EPOCHS, phase_name="phase1")
        fid_scores.append(fid)
        
        if fid < best_seed_fid:
            best_seed_fid = fid
            best_seed_weights = weights
            
    avg_fid = np.mean(fid_scores)
    std_fid = np.std(fid_scores)
    phase1_results.append({'beta': beta, 'mean_fid': avg_fid, 'std_fid': std_fid})
    
    print(f">>> Beta {beta} Avg FID: {avg_fid:.2f} ± {std_fid:.2f}")
    
    if avg_fid < best_beta_fid:
        best_beta_fid = avg_fid
        best_beta = beta
        best_beta_weights = best_seed_weights

print(f"\n[!] Winner of phase 1: Beta = {best_beta}\n")
torch.save(best_beta_weights, "vae_results/best_model_phase1.pth")
print("    [!] Saved best model of Phase 1 to .pth")

pd.DataFrame(phase1_results).to_csv("vae_results/phase1_summary.csv", index=False)

In [ ]:
import shutil

dims_to_test = [32, 64, 128]

print("\n" + "="*50)
print(f" Phase 2: Optimisation of d - base channel width [Using Beta={best_beta}]")
print("="*50)

best_d = None
best_d_fid = float('inf')
best_d_weights = None
phase2_results = []

for d in dims_to_test:
    if d == baseline_d and best_beta == 4.0: 
        print(f"Omitting d={d}, was computed in phase 1.")
        continue 
        
    config = {'beta': best_beta, 'd': d, 'lr': baseline_lr}
    fid_scores = []
    
    best_seed_fid = float('inf')
    best_seed_weights = None
    
    for seed in SEEDS:
        fid, weights = train_and_evaluate(config, seed, train_loader, val_loader, epochs=EPOCHS, phase_name="phase2")
        fid_scores.append(fid)
        
        if fid < best_seed_fid:
            best_seed_fid = fid
            best_seed_weights = weights
            
    avg_fid = np.mean(fid_scores)
    std_fid = np.std(fid_scores)
    phase2_results.append({'d': d, 'mean_fid': avg_fid, 'std_fid': std_fid})
    
    print(f">>> base channel width d={d} Avg FID: {avg_fid:.2f} ± {std_fid:.2f}")
    
    if avg_fid < best_d_fid:
        best_d_fid = avg_fid
        best_d = d
        best_d_weights = best_seed_weights
        
if best_beta == 4.0 and best_beta_fid < best_d_fid:
    best_d = baseline_d
    print(f"\n[!] Winner of phase 2: d = {best_d} (Baseline won! Copying Phase 1 model)")
    shutil.copy("vae_results/best_model_phase1.pth", "vae_results/best_model_phase2.pth")
else:
    print(f"\n[!] Winner of phase 2: d = {best_d}")
    torch.save(best_d_weights, "vae_results/best_model_phase2.pth")
    print("    [!] Saved best model of Phase 2 to .pth")

pd.DataFrame(phase2_results).to_csv("vae_results/phase2_summary.csv", index=False)

In [ ]:
import shutil

lrs_to_test = [1e-4, 1e-3, 1e-2]

print("\n" + "="*50)
print(f" Phase 3: Learning Rate [Using Beta={best_beta}, d={best_d}]")
print("="*50)

best_lr = None
best_lr_fid = float('inf')
best_lr_weights = None
phase3_results = []

for lr in lrs_to_test:
    if lr == baseline_lr:
        print(f"Omitting lr={lr} it was already computed.")
        continue
        
    config = {'beta': best_beta, 'd': best_d, 'lr': lr}
    fid_scores = []
    
    best_seed_fid = float('inf')
    best_seed_weights = None
    
    for seed in SEEDS:
        fid, weights = train_and_evaluate(config, seed, train_loader, val_loader, epochs=EPOCHS, phase_name="phase3")
        fid_scores.append(fid)
        
        if fid < best_seed_fid:
            best_seed_fid = fid
            best_seed_weights = weights
            
    avg_fid = np.mean(fid_scores)
    std_fid = np.std(fid_scores)
    phase3_results.append({'lr': lr, 'mean_fid': avg_fid, 'std_fid': std_fid})
    print(f">>> LR {lr} Avg FID: {avg_fid:.2f} ± {std_fid:.2f}")

    if avg_fid < best_lr_fid:
        best_lr_fid = avg_fid
        best_lr = lr
        best_lr_weights = best_seed_weights

if best_d_fid < best_lr_fid:
    best_lr = baseline_lr
    print(f"\n[!] Winner of phase 3: lr = {best_lr} (Baseline won! Copying Phase 2 model)")
    shutil.copy("vae_results/best_model_phase2.pth", "vae_results/best_model_phase3.pth")
else:
    print(f"\n[!] Winner of phase 3: lr = {best_lr}")
    torch.save(best_lr_weights, "vae_results/best_model_phase3.pth")
    print("    [!] Saved best model of Phase 3 to .pth")

pd.DataFrame(phase3_results).to_csv("vae_results/phase3_summary.csv", index=False)

In [ ]:
best_beta = 1.0
best_d = 32
best_lr = 0.001
epochs = 200

In [ ]:
import numpy as np
import pandas as pd
import torch
import shutil

print("\n" + "="*50)
print(f" FINAL PHASE: Training best architecture on {epochs} epochs")
print(f" Configuration: Beta={best_beta}, d={best_d}, LR={best_lr}")
print("="*50)

config = {'beta': best_beta, 'd': best_d, 'lr': best_lr}
fid_scores = []

best_seed_fid = float('inf')
best_seed_weights = None
final_results = []

for seed in SEEDS:
    fid, weights = train_and_evaluate(config, seed, train_loader, val_loader, epochs=epochs, phase_name=f"final_{epochs}epochs")
    fid_scores.append(fid)
    
    if fid < best_seed_fid:
        best_seed_fid = fid
        best_seed_weights = weights
        
avg_fid = np.mean(fid_scores)
std_fid = np.std(fid_scores)

final_results.append({
    'beta': best_beta, 
    'd': best_d, 
    'lr': best_lr, 
    'mean_fid': avg_fid, 
    'std_fid': std_fid,
    'best_single_fid': best_seed_fid
})

print(f"\n>>> FINAL TRAINING RESULTS: Avg FID over 3 seeds: {avg_fid:.2f} ± {std_fid:.2f}")
print(f">>> Best single seed FID: {best_seed_fid:.2f}")

torch.save(best_seed_weights, "vae_results/best_model_final_500epochs.pth")
print(f"\n    [!] Saved best FINAL model ({epochs} epochs) to 'vae_results/best_model_final_{epochs}epochs.pth'")

pd.DataFrame(final_results).to_csv("vae_results/final_training_summary_500_epoch.csv", index=False)

# Cats vs dogs 

In [ ]:
import os
import zipfile
import shutil


ZIP_PATH = "/kaggle/input/competitions/dogs-vs-cats/train.zip"

EXTRACT_DIR = "/kaggle/working/unzipped_data"
SORTED_DIR = "/kaggle/working/cats_and_dogs_sorted"

if not os.path.exists(EXTRACT_DIR):
    print("Extracting zip files...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Extracting finished!")
else:
    print("FIles are already extracted.")

if not os.path.exists(SORTED_DIR):
    print("Creating folders and sroting photos...")
    cats_dir = os.path.join(SORTED_DIR, "cats")
    dogs_dir = os.path.join(SORTED_DIR, "dogs")
    os.makedirs(cats_dir, exist_ok=True)
    os.makedirs(dogs_dir, exist_ok=True)
    
    train_unzipped_dir = os.path.join(EXTRACT_DIR, "train") 
    
    if not os.path.exists(train_unzipped_dir):
        train_unzipped_dir = EXTRACT_DIR

    moved_cats, moved_dogs = 0, 0
    for filename in os.listdir(train_unzipped_dir):
        file_path = os.path.join(train_unzipped_dir, filename)
        
        if not os.path.isfile(file_path):
            continue
            
        if "cat" in filename.lower():
            shutil.copy(file_path, os.path.join(cats_dir, filename))
            moved_cats += 1
        elif "dog" in filename.lower():
            shutil.copy(file_path, os.path.join(dogs_dir, filename))
            moved_dogs += 1
            
    print(f"Sorted! Cats: {moved_cats}, Dogs: {moved_dogs}")
else:
    print("Photos are sorted and placed in 'cats_and_dogs_sorted'.")

MIXED_DATASET_PATH = SORTED_DIR
print(f"\n[!] Path for pytorch: {MIXED_DATASET_PATH}")

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms


MIXED_DATASET_PATH = "/kaggle/working/cats_and_dogs_sorted" 

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

try:
    mixed_dataset = ImageFolder(root=MIXED_DATASET_PATH, transform=transform)
    print(f"Found {len(mixed_dataset)} images in dataset.")
except Exception as e:
    print(f"Error of loading dataset: {e}")

val_size_mixed = int(0.1 * len(mixed_dataset))
train_size_mixed = len(mixed_dataset) - val_size_mixed
train_dataset_mixed, val_dataset_mixed = random_split(
    mixed_dataset, 
    [train_size_mixed, val_size_mixed], 
    generator=torch.Generator().manual_seed(42)
)

train_loader_mixed = DataLoader(train_dataset_mixed, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
val_loader_mixed = DataLoader(val_dataset_mixed, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
print("\n" + "="*50)
print(f" CATS AND DOGS EXTENSION: Training on {epochs} epochs")
print(f" Configuration: Beta={best_beta}, d={best_d}, LR={best_lr}")
print("="*50)

config = {'beta': best_beta, 'd': best_d, 'lr': best_lr}

fid_scores = []
best_mixed_fid = float('inf')
best_mixed_weights = None
mixed_results = []

for seed in SEEDS:
    fid, weights = train_and_evaluate(
        config, 
        seed, 
        train_loader_mixed, 
        val_loader_mixed, 
        epochs=epochs, 
        phase_name="cats_and_dogs"
    )
    fid_scores.append(fid)
    
    if fid < best_mixed_fid:
        best_mixed_fid = fid
        best_mixed_weights = weights

avg_fid = np.mean(fid_scores)
std_fid = np.std(fid_scores)

mixed_results.append({
    'dataset': 'cats_and_dogs',
    'beta': best_beta, 
    'd': best_d, 
    'lr': best_lr, 
    'mean_fid': avg_fid,
    'std_fid': std_fid,
    'best_single_fid': best_mixed_fid
})

print(f"\n>>> CATS & DOGS FINAL RESULTS: Avg FID over 3 seeds: {avg_fid:.2f} ± {std_fid:.2f}")
print(f">>> Best single seed FID: {best_mixed_fid:.2f}")

torch.save(best_mixed_weights, "vae_results/best_model_cats_and_dogs.pth")
print("\n    [!] Saved best Cats & Dogs model to 'vae_results/best_model_cats_and_dogs.pth'")

pd.DataFrame(mixed_results).to_csv("vae_results/cats_dogs_summary.csv", index=False)